# Following One Prompt Through TorchTitan RL

A short, story-driven tour of `torchtitan/experiments/rl`. We follow a **single
prompt** -- `"2 + 2 = ?"` -- all the way through the GRPO loop, and at every step
we call the **real Titan-RL API** (imported from `torchtitan.experiments.rl`,
not redefined here).

```
  prompt  ->  answers  ->  scored  ->  advantages  ->  training tokens  ->  loss  ->  weight update
              Completion    Rubric      Advantage-      TrainingSample-      GRPOLoss   (buffer bounds
              Rollout       RewardFn    Estimator       Builder                         staleness)
```

Run the cells top to bottom. Only `torch` + the stdlib are required -- the setup
cell handles imports whether or not you have the full vLLM/Monarch stack.


### Setup

Make the local `torchtitan` checkout importable. In a full RL / CI environment the
real deps import directly; otherwise we install a tiny import shim so the
**pure-Python APIs** (types, loss, advantage, rubric, buffer, sample builder) load
without vLLM/Monarch/TorchStore. Either way, the classes you use below are the
**real ones**.


In [1]:
import os, sys

def _find_repo_root(start):
    d = start
    for _ in range(8):
        if os.path.isdir(os.path.join(d, "torchtitan", "experiments", "rl")):
            return d
        nd = os.path.dirname(d)
        if nd == d:
            break
        d = nd
    return None

_root = _find_repo_root(os.path.abspath(os.getcwd()))
if _root and _root not in sys.path:
    sys.path.insert(0, _root)

def _install_import_shim():
    "Fabricate import-time stubs for heavy deps so the pure APIs load. No-op in a full env."
    import importlib.abc, importlib.machinery, logging, types
    ROOTS = ("vllm", "renderers", "monarch", "torchstore", "cloudpickle")
    class _Meta(type):
        def __getattr__(cls, n): return _D
        def __call__(cls, *a, **k):
            if len(a) == 1 and not k and (isinstance(a[0], type) or callable(a[0])): return a[0]
            def deco(t=None): return t if t is not None else _D
            return deco
    class _D(metaclass=_Meta):
        def __init__(self, *a, **k): pass
        def __getattr__(self, n): return _D
        def __call__(self, *a, **k): return _D
    class _Mod(types.ModuleType):
        def __getattr__(self, n):
            if n.startswith("__") and n.endswith("__"): raise AttributeError(n)
            return _D
    class _F(importlib.abc.MetaPathFinder, importlib.abc.Loader):
        def find_spec(self, name, path=None, target=None):
            return importlib.machinery.ModuleSpec(name, self, is_package=True) if name.split(".")[0] in ROOTS else None
        def create_module(self, spec):
            m = _Mod(spec.name); m.__path__ = []; return m
        def exec_module(self, module):
            if module.__name__ == "vllm.logger": module.init_logger = lambda *a, **k: logging.getLogger("stub")
            elif module.__name__ == "renderers": module.Message = dict
    sys.meta_path.insert(0, _F())

try:
    from torchtitan.experiments.rl.rollout.types import RolloutStatus   # probe
    MODE = "full environment (real deps)"
except Exception:
    _install_import_shim()
    from torchtitan.experiments.rl.rollout.types import RolloutStatus   # retry
    MODE = "lightweight (import shim active)"

print("Titan-RL APIs ready:", MODE)


Titan-RL APIs ready: lightweight (import shim active)


## Act 1 - The generator answered our prompt

We asked the policy to solve `"2 + 2 = ?"` and sampled a **group of 4 answers**
(the GRPO group). Each answer comes back as a real `Completion`
(`torchtitan/experiments/rl/types.py`) carrying the generated tokens, their
generator logprobs (needed for the loss later), and the policy version it was
sampled under.


In [2]:
from torchtitan.experiments.rl.types import Completion, RolloutTurnID

prompt_token_ids = [2, 11, 2]                 # pretend-tokenized "2 + 2 = ?"
answers = ["4", "5", "4", "3"]                # what the 4 siblings said (2 correct, 2 wrong)

completions = []
for i, ans in enumerate(answers):
    rid = RolloutTurnID(group_id=0, rollout_id=i, turn_id=0)
    completions.append(Completion(
        min_policy_version=5, max_policy_version=5,       # sampled under policy v5
        request_id=rid.to_string(),                      # "group=0/rollout=0/turn=0"
        token_ids=[100 + int(ans)],                      # the answer token
        token_logprobs=[-0.2],
        finish_reason="stop",
    ))

for c in completions:
    print(f"{c.request_id:26} tokens={c.token_ids}  logprobs={c.token_logprobs}  v={c.min_policy_version}")


group=0/rollout=0/turn=0   tokens=[104]  logprobs=[-0.2]  v=5
group=0/rollout=1/turn=0   tokens=[105]  logprobs=[-0.2]  v=5
group=0/rollout=2/turn=0   tokens=[104]  logprobs=[-0.2]  v=5
group=0/rollout=3/turn=0   tokens=[103]  logprobs=[-0.2]  v=5


## Act 2 - Wrap answers into a `RolloutGroup`

The rollout loop turns each `Completion` into a `RolloutTurn` (the full per-turn
snapshot), bundles a rollout's turns into a `Rollout`, and the 4 siblings into a
`RolloutGroup` -- all real types from `rollout/types.py`. (A multi-turn task would
have several `RolloutTurn`s per `Rollout`; ours is single-turn.)


In [3]:
from torchtitan.experiments.rl.rollout.types import Rollout, RolloutTurn, RolloutGroup, RolloutStatus

rollouts = []
for i, (ans, c) in enumerate(zip(answers, completions)):
    turn = RolloutTurn(
        rollout_id=RolloutTurnID(group_id=0, rollout_id=i, turn_id=0),
        prompt_token_ids=prompt_token_ids,
        completion_token_ids=c.token_ids,
        completion_logprobs=c.token_logprobs,
        completion_message={"role": "assistant", "content": ans},
        min_policy_version=5, max_policy_version=5,
    )
    rollouts.append(Rollout(group_id=0, rollout_id=i, status=RolloutStatus.COMPLETED, turns=[turn]))

group = RolloutGroup(group_id=0, rollouts=rollouts)
print("RolloutGroup with", len(group.rollouts), "siblings, all", group.rollouts[0].status.value)
print("answers:", [r.turns[-1].completion_message["content"] for r in group.rollouts])


RolloutGroup with 4 siblings, all completed
answers: ['4', '5', '4', '3']


## Act 3 - Score the group with a `Rubric`

Scoring is a real extension point: you write a `RewardFn` subclass and hand it to
a `Rubric` (`rubrics/rubric.py`). Here a tiny correctness reward gives 1.0 for the
right answer. This is exactly how you'd score your own task.


In [4]:
import asyncio
from dataclasses import dataclass
from torchtitan.experiments.rl.rubrics import RewardFn, Rubric

class CorrectnessReward(RewardFn):
    @dataclass(kw_only=True, slots=True)          # each RewardFn declares its own Config
    class Config(RewardFn.Config):
        pass
    async def __call__(self, rollout, env_input) -> float:
        said = rollout.turns[-1].completion_message["content"].strip()
        return 1.0 if said == env_input else 0.0

target = "4"
rubric = Rubric.Config(reward_fns=[CorrectnessReward.Config(weight=1.0)]).build()
outputs = await rubric.score_group(group.rollouts, target)

for r, out in zip(group.rollouts, outputs):
    r.reward = out.reward
    r.reward_breakdown = out.reward_breakdown
print("rewards:", [r.reward for r in group.rollouts], " (answers were", answers, "vs target", repr(target) + ")")


rewards: [1.0, 0.0, 1.0, 0.0]  (answers were ['4', '5', '4', '3'] vs target '4')


## Act 4 - Turn rewards into advantages

`AdvantageEstimator` (`rollout/advantage.py`) centers the rewards within the group
-- an answer that beat its siblings gets a positive advantage. Default is Dr.GRPO
(mean baseline). We attach each advantage back onto its `Rollout`.


In [5]:
from torchtitan.experiments.rl.rollout.advantage import AdvantageEstimator

estimator = AdvantageEstimator.Config().build()      # Dr.GRPO (mean baseline)
advantages = estimator(group)                        # reads r.reward, returns one per rollout

for r, a in zip(group.rollouts, advantages):
    r.advantage = a
print("rewards   :", [r.reward for r in group.rollouts])
print("advantages:", [round(r.advantage, 3) for r in group.rollouts], " (mean-centered: correct > 0, wrong < 0)")


rewards   : [1.0, 0.0, 1.0, 0.0]
advantages: [0.5, -0.5, 0.5, -0.5]  (mean-centered: correct > 0, wrong < 0)


## Act 5 - Pack the rollout into trainable tokens

`TrainingSampleBuilder` (`components/training_sample_builder.py`) converts the
scored group into `TrainingSample`s: it drops untrainable / zero-variance groups,
then lays out `token_ids`, a `loss_mask` (True only on tokens to train), the
generator `logprobs`, and the per-token `advantage`. Multi-turn rollouts get
packed (or branched) here too.


In [6]:
from torchtitan.experiments.rl.components.training_sample_builder import TrainingSampleBuilder

builder = TrainingSampleBuilder.Config().build()
tsg = builder.build_from_group(rollout_group=group)   # -> TrainingSampleGroup

print("training samples produced:", len(tsg.training_samples), "(group survived: rewards have variance)\n")
s = tsg.training_samples[0]
print("one TrainingSample (rollout 0):")
print("  token_ids :", s.token_ids, "  <- prompt + completion")
print("  loss_mask :", s.loss_mask, "  <- train only the completion token(s)")
print("  logprobs  :", s.logprobs,  "  <- generator logprobs, 0.0 on prompt")
print("  advantage :", s.advantage, "  <- broadcast onto trained tokens")


training samples produced: 4 (group survived: rewards have variance)

one TrainingSample (rollout 0):
  token_ids : [2, 11, 2, 104]   <- prompt + completion
  loss_mask : [False, False, False, True]   <- train only the completion token(s)
  logprobs  : [0.0, 0.0, 0.0, -0.2]   <- generator logprobs, 0.0 on prompt
  advantage : [0.0, 0.0, 0.0, 0.5]   <- broadcast onto trained tokens


## Act 6 - The loss turns tokens into a gradient signal

Finally the real `GRPOLoss` (`losses/grpo.py`) consumes that sample. It compares
the *trainer's* logprobs (from the model's `logits`) against the *generator's*
logprobs, forms the clipped importance ratio, weights by advantage, and returns a
scalar loss (+ diagnostic metrics). We fabricate `logits` here since we have no
model in this notebook.


In [7]:
import torch
from torchtitan.experiments.rl.losses import GRPOLoss

loss_fn = GRPOLoss.Config(clip_eps=0.2).build()

s = tsg.training_samples[0]
L, V = len(s.token_ids), 128
logits = torch.zeros(1, L, V)                          # stand-in for model(token_ids)
labels = torch.tensor([s.token_ids])
global_valid_tokens = sum(s.loss_mask)                 # denominator = # trained tokens

loss, metrics = loss_fn(
    logits, labels, global_valid_tokens,
    generator_logprobs=torch.tensor([s.logprobs]),
    advantages=torch.tensor([s.advantage]),
    loss_mask=torch.tensor([s.loss_mask]),
)
print("GRPO loss:", round(loss.item(), 4))
print("clipped fraction:", round(metrics["loss/ratio_clipped_frac"].item(), 4))
print("\nThat scalar is what .backward() runs on -> the optimizer nudges the policy.")


GRPO loss: -0.0048
clipped fraction: 1.0

That scalar is what .backward() runs on -> the optimizer nudges the policy.


## Act 7 - Off-policy control: the real rollout buffer

One more real API. The `Controller` runs generation ahead of training, but not too
far: the `RolloutGroupWorkBuffer` (`components/work_buffer.py`) caps how many
groups are in flight. When the window is full, the data loop blocks until a
training step releases slots -- keeping training data fresh.


In [8]:
from torchtitan.experiments.rl.components.work_buffer import RolloutGroupWork, RolloutGroupWorkBuffer

async def buffer_story():
    buf = RolloutGroupWorkBuffer.Config().build(max_active_rollout_groups=2)   # window = 2 groups

    ok0 = await buf.wait_for_slot(); await buf.add_work(RolloutGroupWork(group_id=0, sample=object()))
    ok1 = await buf.wait_for_slot(); await buf.add_work(RolloutGroupWork(group_id=1, sample=object()))
    print(f"acquired 2 slots (window full): {ok0} {ok1}")

    third = asyncio.create_task(buf.wait_for_slot())      # 3rd producer blocks
    await asyncio.sleep(0.05)
    print("3rd wait_for_slot done yet?", third.done(), " <- blocked: generation cannot outrun the window")

    await buf.release_active_groups(1, reason="trained")  # a train step frees one slot
    await third
    print("after a training step released a slot, the 3rd producer proceeds:", third.done())

await buffer_story()


acquired 2 slots (window full): True True
3rd wait_for_slot done yet? False  <- blocked: generation cannot outrun the window
after a training step released a slot, the 3rd producer proceeds: True


## Act 8 - Where the actors live (needs a GPU cluster)

Acts 1-7 ran the real *data-plane* APIs on your laptop. The *compute* actors need
GPUs + vLLM + Monarch, so here are their real signatures (see the source):

- **`VLLMGenerator`** (`actors/generator.py`) produced Act 1's `Completion`s:
  `await generator.generate.call_one(prompt_token_ids, request_id=..., routing_session_id=...)`
- **`PolicyTrainer`** (`actors/trainer.py`) consumes Act 6's loss:
  `await trainer.forward_backward(microbatch, num_global_valid_tokens)` then
  `await trainer.optim_step()` then `await trainer.push_model_state_dict()`
- **`Controller`** (`controller.py`) wires it all: 4 async loops
  (`_data_input_loop -> _rollout_loop -> _batcher_loop -> _trainer_loop`) plus the
  buffer from Act 7 and weight sync via TorchStore.

Run the real thing (needs the RL env + a GPU):
```bash
python -m torchtitan.experiments.rl.train --module alphabet_sort \
  --config rl_grpo_qwen3_0_6b_varlen --metrics.no-enable-wandb
```


## The end - recap and where to go next

You just followed one prompt through the real pipeline:

| Act | Real API | Turned ... into ... |
|---|---|---|
| 1 | `Completion` | prompt -> sampled answers |
| 2 | `Rollout` / `RolloutGroup` | answers -> a scored-ready group |
| 3 | `Rubric` + `RewardFn` | answers -> rewards |
| 4 | `AdvantageEstimator` | rewards -> advantages |
| 5 | `TrainingSampleBuilder` | rollout -> training tokens |
| 6 | `GRPOLoss` | tokens -> a gradient signal |
| 7 | `RolloutGroupWorkBuffer` | (bounds how stale data may get) |
| 8 | `VLLMGenerator` / `PolicyTrainer` / `Controller` | the compute actors |

**Read next, in order:** `types.py` + `rollout/types.py` (Acts 1-2) ->
`rubrics/rubric.py`, `rollout/advantage.py` (Acts 3-4) ->
`components/training_sample_builder.py`, `losses/dapo.py` (Acts 5-6) ->
`components/work_buffer.py`, `controller.py`, `actors/*` (Acts 7-8).

Deeper dives live alongside this notebook: `docs/test_plan.md` (how it is tested),
`docs/rl_architecture_and_forge_comparison.md` (full API reference),
`docs/rl_frameworks_illustrated.md` (Titan vs Forge vs Slime).
